
# Caso de clase CAPM — Parte 1
## Del retorno observado al beta: acciones, ETF y fondos

Este notebook estudia directamente el **Capital Asset Pricing Model (CAPM)** sin pasar por Markowitz.

La idea pedagógica es comparar activos con características muy diferentes:

- acciones de alta volatilidad;
- acciones de precio alto;
- acciones de precio bajo;
- acciones defensivas;
- ETF;
- fondos de inversión.

La pregunta central es:

> **¿Qué determina el beta y por qué el precio de una acción no es una medida de riesgo sistemático?**

El CAPM establece:

$$
E(R_i)-R_f
=
\beta_i\left[E(R_m)-R_f\right]
$$

donde:

- $R_i$ es el retorno del activo;
- $R_f$ es la tasa libre de riesgo;
- $R_m$ es el retorno del mercado;
- $\beta_i$ mide la exposición del activo al riesgo sistemático.


In [ ]:
# Si hace falta instalar:
# !pip install yfinance pandas numpy matplotlib statsmodels scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Selección de activos

In [ ]:
start_date = "2018-01-01"
end_date   = "2025-12-31"

# Acciones elegidas para contrastar precio, volatilidad y beta
stocks = {
    "Tesla": "TSLA",          # alta volatilidad
    "Nvidia": "NVDA",         # alta volatilidad / crecimiento
    "Berkshire_B": "BRK-B",   # precio relativamente alto, perfil distinto
    "Booking": "BKNG",        # precio nominal muy alto
    "Ford": "F",              # precio nominal bajo
    "Snap": "SNAP",           # precio nominal bajo y volátil
    "CocaCola": "KO",         # defensiva
    "JPMorgan": "JPM"         # financiera cíclica
}

etfs = {
    "SP500": "SPY",           # proxy de mercado
    "Nasdaq100": "QQQ",
    "SmallCap": "IWM",
    "Technology": "XLK",
    "Healthcare": "XLV",
    "Energy": "XLE"
}

funds = {
    "Vanguard500Fund": "VFIAX",
    "VanguardTotalMarket": "VTSAX",
    "FidelityContrafund": "FCNTX"
}

all_assets = {**stocks, **etfs, **funds}
tickers = list(all_assets.values())

all_assets

## 2. Descarga robusta de precios

Se descarga cada activo individualmente. Esto evita que un ticker con datos faltantes impida trabajar con el resto del caso.


In [ ]:
def download_close(ticker, start, end):
    data = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    if data.empty:
        return None

    close = data["Close"]

    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    close.name = ticker
    return close

series = []

for ticker in tickers:
    s = download_close(ticker, start_date, end_date)
    if s is not None:
        series.append(s)

prices_raw = pd.concat(series, axis=1)

ticker_to_name = {v: k for k, v in all_assets.items()}
prices = prices_raw.rename(columns=ticker_to_name)

print("Activos descargados:")
print(list(prices.columns))
prices.tail()

## 3. Precio nominal: una primera intuición engañosa

In [ ]:
latest_prices = prices.ffill().iloc[-1].sort_values(ascending=False)

latest_price_table = pd.DataFrame({
    "Latest_Price": latest_prices
})

latest_price_table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

latest_prices.sort_values().plot(kind="barh", ax=ax)

ax.set_title("Precio nominal más reciente de cada activo")
ax.set_xlabel("Precio")
ax.set_ylabel("Activo")

plt.tight_layout()
plt.show()


### Pregunta para clase

¿Una acción que cuesta 3.000 dólares es necesariamente más riesgosa que una acción que cuesta 10 dólares?

**No.** El precio nominal no determina el riesgo sistemático.

El beta depende del co-movimiento con el mercado:

$$
\beta_i
=
\frac{\operatorname{Cov}(R_i,R_m)}
{\operatorname{Var}(R_m)}
$$

Por eso, una acción de precio alto puede tener beta bajo y una acción barata puede tener beta alto.


## 4. Retornos mensuales

In [ ]:
monthly_prices = prices.resample("ME").last()

returns = monthly_prices.pct_change(fill_method=None)

# Se conserva cada activo con sus observaciones disponibles.
returns.tail()

## 5. Retorno y volatilidad: todavía no estamos midiendo beta

In [ ]:
summary = pd.DataFrame({
    "Mean_Monthly_Return": returns.mean(),
    "Monthly_Volatility": returns.std(),
    "Annualized_Return": returns.mean() * 12,
    "Annualized_Volatility": returns.std() * np.sqrt(12),
    "Latest_Price": prices.ffill().iloc[-1]
})

summary.sort_values("Annualized_Volatility", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

valid = summary.dropna()

ax.scatter(
    valid["Annualized_Volatility"],
    valid["Annualized_Return"]
)

for asset in valid.index:
    ax.annotate(
        asset,
        (valid.loc[asset, "Annualized_Volatility"],
         valid.loc[asset, "Annualized_Return"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.set_xlabel("Volatilidad anualizada")
ax.set_ylabel("Retorno promedio anualizado")
ax.set_title("Riesgo total vs. retorno")

plt.tight_layout()
plt.show()


La volatilidad mide **riesgo total**:

$$
\sigma_i
=
\sqrt{\operatorname{Var}(R_i)}
$$

El CAPM, en cambio, no afirma que toda la volatilidad sea remunerada. Su variable central es el **riesgo sistemático**, resumido por $\beta$.


## 6. Tasa libre de riesgo y retornos en exceso

In [ ]:
# Para mantener el notebook reproducible:
rf_annual = 0.04
rf_monthly = (1 + rf_annual)**(1/12) - 1

market = "SP500"

excess_returns = returns - rf_monthly

print("Rf anual:", rf_annual)
print("Rf mensual equivalente:", rf_monthly)


La regresión empírica del CAPM es:

$$
R_{i,t}-R_{f,t}
=
\alpha_i
+
\beta_i\left(R_{m,t}-R_{f,t}\right)
+
\varepsilon_{i,t}
$$

Bajo la versión estricta del CAPM:

$$
\alpha_i = 0
$$

y el retorno esperado debería venir explicado por la exposición al mercado.


## 7. Función CAPM para cualquier activo

In [ ]:
def estimate_capm(asset, excess_returns, market="SP500"):
    data = pd.concat(
        [
            excess_returns[asset].rename("asset"),
            excess_returns[market].rename("market")
        ],
        axis=1
    ).dropna()

    y = data["asset"]
    X = sm.add_constant(data["market"])

    model = sm.OLS(y, X).fit()

    beta_cov = (
        np.cov(data["asset"], data["market"], ddof=1)[0, 1]
        /
        np.var(data["market"], ddof=1)
    )

    result = {
        "Asset": asset,
        "N": len(data),
        "Alpha_monthly": model.params["const"],
        "Beta": model.params["market"],
        "Beta_covariance": beta_cov,
        "R2": model.rsquared,
        "Alpha_pvalue": model.pvalues["const"],
        "Beta_pvalue": model.pvalues["market"]
    }

    return result, model

## 8. Primer contraste: Tesla, Booking, Ford y Coca-Cola

In [ ]:
comparison_assets = [
    a for a in ["Tesla", "Booking", "Ford", "CocaCola"]
    if a in excess_returns.columns
]

rows = []
models = {}

for asset in comparison_assets:
    result, model = estimate_capm(asset, excess_returns, market)
    rows.append(result)
    models[asset] = model

comparison_capm = pd.DataFrame(rows).set_index("Asset")
comparison_capm


Este bloque permite discutir cuatro perfiles:

- **Tesla:** alta volatilidad;
- **Booking:** precio nominal alto;
- **Ford:** precio nominal bajo;
- **Coca-Cola:** comportamiento defensivo.

La pregunta es si el orden de los precios coincide con el orden de los betas.

No tendría por qué ocurrir, porque:

$$
\text{Precio nominal}
\neq
\text{Volatilidad}
\neq
\beta
$$


## 9. Rectas CAPM: comparación visual

In [ ]:
for asset in comparison_assets:

    data = pd.concat(
        [
            excess_returns[asset].rename("asset"),
            excess_returns[market].rename("market")
        ],
        axis=1
    ).dropna()

    model = models[asset]

    x_grid = np.linspace(data["market"].min(), data["market"].max(), 100)
    y_grid = model.params["const"] + model.params["market"] * x_grid

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(data["market"], data["asset"], alpha=0.6)
    ax.plot(x_grid, y_grid)

    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)

    ax.set_title(
        f"{asset} vs. mercado | beta = {model.params['market']:.2f}"
    )
    ax.set_xlabel("Exceso de retorno del mercado")
    ax.set_ylabel(f"Exceso de retorno de {asset}")

    plt.tight_layout()
    plt.show()

## 10. CAPM para todas las acciones

In [ ]:
stock_results = []

for asset in stocks.keys():
    if asset in excess_returns.columns:
        result, _ = estimate_capm(asset, excess_returns, market)
        stock_results.append(result)

stock_capm = pd.DataFrame(stock_results).set_index("Asset")

stock_table = stock_capm.join(
    summary[["Latest_Price", "Annualized_Volatility"]]
)

stock_table.sort_values("Beta", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

beta_sorted = stock_table["Beta"].sort_values()

beta_sorted.plot(kind="barh", ax=ax)
ax.axvline(1, linewidth=1)

ax.set_title("Beta de acciones con perfiles muy diferentes")
ax.set_xlabel("Beta")
ax.set_ylabel("Acción")

plt.tight_layout()
plt.show()


### Lectura de beta

$$
\beta_i > 1
$$

indica una respuesta más que proporcional a los movimientos del mercado.

$$
0 < \beta_i < 1
$$

indica menor sensibilidad que el mercado.

$$
\beta_i \approx 1
$$

indica sensibilidad similar al mercado.


## 11. Precio vs. beta

In [ ]:
price_beta = stock_table.dropna(subset=["Latest_Price", "Beta"])

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    price_beta["Latest_Price"],
    price_beta["Beta"]
)

for asset in price_beta.index:
    ax.annotate(
        asset,
        (price_beta.loc[asset, "Latest_Price"],
         price_beta.loc[asset, "Beta"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.axhline(1, linewidth=1)
ax.set_xlabel("Precio nominal")
ax.set_ylabel("Beta")
ax.set_title("Precio nominal vs. riesgo sistemático")

plt.tight_layout()
plt.show()


La gráfica permite mostrar directamente que el beta no depende del nivel de precio.

El beta está definido por:

$$
\beta_i
=
\frac{\operatorname{Cov}(R_i,R_m)}
{\operatorname{Var}(R_m)}
$$

Por tanto, importa **cómo se mueve el retorno del activo con el mercado**, no cuánto cuesta una unidad de la acción.


## 12. Volatilidad vs. beta

In [ ]:
vol_beta = stock_table.dropna(subset=["Annualized_Volatility", "Beta"])

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    vol_beta["Annualized_Volatility"],
    vol_beta["Beta"]
)

for asset in vol_beta.index:
    ax.annotate(
        asset,
        (vol_beta.loc[asset, "Annualized_Volatility"],
         vol_beta.loc[asset, "Beta"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.axhline(1, linewidth=1)
ax.set_xlabel("Volatilidad anualizada")
ax.set_ylabel("Beta")
ax.set_title("Volatilidad total vs. beta")

plt.tight_layout()
plt.show()


Una acción puede tener alta volatilidad sin que toda esa volatilidad provenga del mercado.

En la regresión CAPM:

$$
R_i-R_f
=
\alpha_i
+
\beta_i(R_m-R_f)
+
\varepsilon_i
$$

el término:

$$
\beta_i(R_m-R_f)
$$

representa el componente asociado al mercado, mientras que:

$$
\varepsilon_i
$$

captura movimientos idiosincráticos.


## 13. CAPM para ETF

In [ ]:
etf_results = []

for asset in etfs.keys():
    if asset in excess_returns.columns and asset != market:
        result, _ = estimate_capm(asset, excess_returns, market)
        etf_results.append(result)

etf_capm = pd.DataFrame(etf_results).set_index("Asset")
etf_capm.sort_values("Beta", ascending=False)


Los ETF permiten ver exposiciones sistemáticas más limpias porque representan portafolios diversificados.

Por ejemplo:

- QQQ y XLK deberían reflejar exposición a tecnología;
- IWM representa small caps;
- XLV sectores defensivos de salud;
- XLE exposición energética.

El CAPM permite comparar si estas exposiciones producen betas sistemáticamente distintos.


## 14. CAPM para fondos de inversión

In [ ]:
fund_results = []

for asset in funds.keys():
    if asset in excess_returns.columns:
        result, _ = estimate_capm(asset, excess_returns, market)
        fund_results.append(result)

if fund_results:
    fund_capm = pd.DataFrame(fund_results).set_index("Asset")
    display(fund_capm.sort_values("Beta", ascending=False))
else:
    print("No se descargaron fondos. Yahoo Finance puede variar en disponibilidad.")

## 15. Comparación conjunta: acciones, ETF y fondos

In [ ]:
combined_rows = []

for asset in stocks.keys():
    if asset in excess_returns.columns:
        result, _ = estimate_capm(asset, excess_returns, market)
        result["Type"] = "Stock"
        combined_rows.append(result)

for asset in etfs.keys():
    if asset in excess_returns.columns and asset != market:
        result, _ = estimate_capm(asset, excess_returns, market)
        result["Type"] = "ETF"
        combined_rows.append(result)

for asset in funds.keys():
    if asset in excess_returns.columns:
        result, _ = estimate_capm(asset, excess_returns, market)
        result["Type"] = "Fund"
        combined_rows.append(result)

combined_capm = pd.DataFrame(combined_rows).set_index("Asset")
combined_capm.sort_values(["Type", "Beta"], ascending=[True, False])


# Conclusión de la Parte 1

El objetivo no es concluir que un activo es riesgoso porque:

- cuesta mucho;
- cuesta poco;
- tiene una gran volatilidad.

El CAPM propone una definición específica de riesgo relevante:

$$
\boxed{
\text{Riesgo sistemático}
\longrightarrow
\beta
}
$$

y no:

$$
\boxed{
\text{Precio nominal}
\longrightarrow
\text{riesgo}
}
$$

La Parte 2 estudia si ese beta realmente explica los retornos observados y cómo cambia la interpretación al comparar acciones, ETF y fondos.
